# 🦉 Kiểm tra và Xác thực Ánh xạ Loài RFCx (BioListen VN)

Notebook này được thiết kế để giúp bạn **nghe thử âm thanh thực tế** của 24 loài sinh vật trong tập dữ liệu Rainforest Connection (RFCx) từ `s0` đến `s23` để xác minh danh tính sinh học của chúng (Chim hay Ếch).

### Nguồn âm thanh:
Chúng ta truy vấn trực tiếp thư viện âm thanh sinh học toàn cầu **Xeno-canto** thông qua API mở (không cần key). Bạn có thể bấm nút **Play** trực tiếp trên Notebook để nghe tiếng kêu của loài mà không cần tải hàng chục GB dữ liệu thô.

In [ ]:
import requests
from IPython.display import display, HTML, Audio

# Từ điển ánh xạ 24 loài gốc vùng Puerto Rico từ s0 -> s23
RFCX_SPECIES_MAP = {
    0: {"id": "s0", "scientific": "Eleutherodactylus coqui", "english": "Common Coqui", "vietnamese": "Ếch Coquí thông thường", "type": "Ếch (Frog)"},
    1: {"id": "s1", "scientific": "Megascops nudipes", "english": "Puerto Rican Screech-Owl", "vietnamese": "Cú mèo Puerto Rico", "type": "Chim (Bird)"},
    2: {"id": "s2", "scientific": "Todus mexicanus", "english": "Puerto Rican Tody", "vietnamese": "Chim Tody Puerto Rico", "type": "Chim (Bird)"},
    3: {"id": "s3", "scientific": "Coereba flaveola", "english": "Bananaquit", "vietnamese": "Chim Bananaquit", "type": "Chim (Bird)"},
    4: {"id": "s4", "scientific": "Melanerpes portoricensis", "english": "Puerto Rican Woodpecker", "vietnamese": "Gõ kiến Puerto Rico", "type": "Chim (Bird)"},
    5: {"id": "s5", "scientific": "Loxigilla portoricensis", "english": "Puerto Rican Bullfinch", "vietnamese": "Chim sẻ thông Puerto Rico", "type": "Chim (Bird)"},
    6: {"id": "s6", "scientific": "Vireo latimeri", "english": "Puerto Rican Vireo", "vietnamese": "Chim Vireo Puerto Rico", "type": "Chim (Bird)"},
    7: {"id": "s7", "scientific": "Spindalis portoricensis", "english": "Puerto Rican Spindalis", "vietnamese": "Chim Spindalis Puerto Rico", "type": "Chim (Bird)"},
    8: {"id": "s8", "scientific": "Crotophaga ani", "english": "Smooth-billed Ani", "vietnamese": "Chim Ani mỏ nhẵn", "type": "Chim (Bird)"},
    9: {"id": "s9", "scientific": "Buteo platypterus", "english": "Broad-winged Hawk", "vietnamese": "Diều hâu cánh rộng", "type": "Chim (Bird)"},
    10: {"id": "s10", "scientific": "Leptodactylus albilabris", "english": "White-lipped Frog", "vietnamese": "Ếch môi trắng", "type": "Ếch (Frog)"},
    11: {"id": "s11", "scientific": "Eleutherodactylus antillensis", "english": "Red-eyed Coqui", "vietnamese": "Ếch mắt đỏ", "type": "Ếch (Frog)"},
    12: {"id": "s12", "scientific": "Eleutherodactylus brittoni", "english": "Britton's Coqui", "vietnamese": "Ếch Britton", "type": "Ếch (Frog)"},
    13: {"id": "s13", "scientific": "Eleutherodactylus wightmanae", "english": "Wightman's Coqui", "vietnamese": "Ếch Wightman", "type": "Ếch (Frog)"},
    14: {"id": "s14", "scientific": "Eleutherodactylus richmondi", "english": "Richmond's Coqui", "vietnamese": "Ếch Richmond", "type": "Ếch (Frog)"},
    15: {"id": "s15", "scientific": "Eleutherodactylus gryllus", "english": "Cricket Coqui", "vietnamese": "Ếch dế", "type": "Ếch (Frog)"},
    16: {"id": "s16", "scientific": "Eleutherodactylus locustus", "english": "Locust Coqui", "vietnamese": "Ếch châu chấu", "type": "Ếch (Frog)"},
    17: {"id": "s17", "scientific": "Eleutherodactylus hedricki", "english": "Hedrick's Coqui", "vietnamese": "Ếch Hedrick", "type": "Ếch (Frog)"},
    18: {"id": "s18", "scientific": "Eleutherodactylus unicolor", "english": "Bronze Coqui", "vietnamese": "Ếch đồng", "type": "Ếch (Frog)"},
    19: {"id": "s19", "scientific": "Eleutherodactylus portoricensis", "english": "Puerto Rican Coqui", "vietnamese": "Ếch rừng Puerto Rico", "type": "Ếch (Frog)"},
    20: {"id": "s20", "scientific": "Eleutherodactylus cooki", "english": "Guajón Coqui", "vietnamese": "Ếch đá", "type": "Ếch (Frog)"},
    21: {"id": "s21", "scientific": "Eleutherodactylus eneidae", "english": "Eneida's Coqui", "vietnamese": "Ếch Eneida", "type": "Ếch (Frog)"},
    22: {"id": "s22", "scientific": "Eleutherodactylus karlschmidti", "english": "Karlschmidt's Coqui", "vietnamese": "Ếch suối Karlschmidt", "type": "Ếch (Frog)"},
    23: {"id": "s23", "scientific": "Lithobates catesbeianus", "english": "American Bullfrog", "vietnamese": "Ếch ương lớn", "type": "Ếch (Frog)"}
}

print(f"✓ Loaded dictionary mapping for {len(RFCX_SPECIES_MAP)} species.")

### Hàm kết nối API Xeno-canto để truy vấn âm thanh

In [ ]:
def get_xeno_canto_audio_url(scientific_name):
    """
    Gửi request đến API Xeno-canto để tìm bản ghi âm đầu tiên của loài.
    """
    api_url = f"https://xeno-canto.org/api/2/recordings?query={scientific_name.replace(' ', '%20')}"
    try:
        resp = requests.get(api_url, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            recordings = data.get('recordings', [])
            if recordings:
                # Lấy bản ghi đầu tiên có link download
                file_url = recordings[0].get('file')
                if file_url:
                    if file_url.startswith('//'):
                        file_url = 'https:' + file_url
                    return file_url
    except Exception as e:
        print(f"Lỗi kết nối API cho loài {scientific_name}: {e}")
    return None

### Trình phát âm thanh & Thông tin chi tiết theo ID

In [ ]:
def play_species(id_str):
    """
    Hàm hiển thị thông tin và phát âm thanh mẫu từ ID dạng s0 - s23.
    """
    id_str = id_str.strip().lower()
    # Tìm kiếm theo id string
    target_key = None
    for k, v in RFCX_SPECIES_MAP.items():
        if v['id'] == id_str:
            target_key = k
            break
            
    if target_key is None:
        # Nếu nhập số nguyên
        try:
            target_key = int(id_str)
        except ValueError:
            pass
            
    if target_key not in RFCX_SPECIES_MAP:
        print(f"❌ Không tìm thấy loài tương ứng với nhãn '{id_str}' (Vui lòng nhập s0 -> s23)")
        return
        
    info = RFCX_SPECIES_MAP[target_key]
    
    # Thiết lập giao diện HTML hiển thị
    html_template = f"""
    <div style='border: 2px solid #4CAF50; border-radius: 8px; padding: 15px; background-color: #f9f9f9; max-width: 600px; font-family: Arial, sans-serif;'>
        <h3 style='margin-top: 0; color: #2E7D32;'>🔎 CHI TIẾT LOÀI: {info['id'].upper()}</h3>
        <table style='width: 100%; border-collapse: collapse;'>
            <tr style='border-bottom: 1px solid #ddd;'>
                <td style='padding: 8px 0; font-weight: bold; width: 40%;'>Tên khoa học:</td>
                <td style='padding: 8px 0; font-style: italic;'>{info['scientific']}</td>
            </tr>
            <tr style='border-bottom: 1px solid #ddd;'>
                <td style='padding: 8px 0; font-weight: bold;'>Phân nhóm:</td>
                <td style='padding: 8px 0;'>{info['type']}</td>
            </tr>
            <tr style='border-bottom: 1px solid #ddd;'>
                <td style='padding: 8px 0; font-weight: bold;'>Tên tiếng Anh:</td>
                <td style='padding: 8px 0;'>{info['english']}</td>
            </tr>
            <tr>
                <td style='padding: 8px 0; font-weight: bold;'>Tên tiếng Việt đề xuất:</td>
                <td style='padding: 8px 0; color: #D32F2F; font-weight: bold;'>{info['vietnamese']}</td>
            </tr>
        </table>
    </div>
    """
    display(HTML(html_template))
    
    print("Đang kết nối đến thư viện âm thanh Xeno-canto...")
    audio_url = get_xeno_canto_audio_url(info['scientific'])
    if audio_url:
        display(Audio(url=audio_url))
    else:
        print("❌ Không tìm thấy bản ghi âm hoặc API quá tải. Vui lòng thử lại sau.")

## 🎧 PHẦN KIỂM TRA & XÁC THỰC ÂM THANH

Bạn có thể thay đổi tham số truyền vào hàm `play_species` bên dưới (từ `s0` đến `s23`) và bấm **Run Cell** để nghe thử tiếng kêu của từng loài và xác minh tên Anh/Việt.

In [ ]:
# Nghe tiếng ếch s0 (Common Coqui)
play_species('s0')

In [ ]:
# Nghe tiếng cú mèo s1 (Puerto Rican Screech-Owl)
play_species('s1')

In [ ]:
# Nghe tiếng chim Tody s2 (Puerto Rican Tody)
play_species('s2')

In [ ]:
# Chạy đoạn này để in ra toàn bộ checklist giúp bạn tra cứu nhanh
print("=== BẢNG CHỈ DẪN 24 LOÀI ===")
for k, v in RFCX_SPECIES_MAP.items():
    print(f"{v['id'].upper()} -> {v['scientific']} ({v['type']}) | Anh: {v['english']} | Việt: {v['vietnamese']}")